# Week 4 Task 4 — Approach A: Fare Training Dataset Built from Raw Files

## Goal

Rebuild the Week 4 Task 1 fare training dataset **without the data platform**, starting only from the original files in `data/raw/`, and verify that the result is identical to the platform-based dataset produced by `w4_task1.ipynb` (Approach B).

From the feature pipeline onward, Approach A and Approach B must use the same code. Any difference in effort, run time, or reproducibility can then be attributed to the presence or absence of the Week 1–3 platform.

## Boundary

| | Allowed | Not allowed |
|---|---|---|
| Input | The six files in `data/raw/` | Any Delta table under `data/processed/` |
| Code | `build_spark_session()` (execution environment only) | Functions from `ingestion_framework.ipynb` or `integrated_taxi_trips.ipynb` |
| Output | `data/processed/ml_raw/fare_training_dataset` | Overwriting the Approach B dataset |

The platform dataset is read **only** in the final equivalence check, which is disabled during timing runs.

## Strategy: faithful reproduction

Every platform rule that affects the training rows is re-implemented here: column renaming and typing, the `trip_id` hash, the taxi, weather, air-quality and zone validation rules, primary-key deduplication, the zone/weather/air-quality joins, and the Task 1 selection, feature, and split logic.

## Setup

The Spark session uses the same configuration as `w4_task1.ipynb`, including the `America/New_York` session time zone. The time zone matters because naive raw timestamps are interpreted in the session zone, and `trip_id` hashes the string form of those timestamps.

In [1]:
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
import json
import math
import os
import sys
import time
import uuid

from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import DoubleType, IntegerType, LongType, TimestampType
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TAXI_SOURCE = RAW_DIR / "yellow_tripdata_2024-*.parquet"
ZONE_SOURCE = RAW_DIR / "taxi_zone_lookup.csv"
WEATHER_SOURCE = RAW_DIR / "weather.csv"
AIR_QUALITY_SOURCE = RAW_DIR / "hourly_88101_2024.csv"

OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "ml_raw" / "fare_training_dataset"
RUN_MANIFEST_PATH = PROJECT_ROOT / "artifacts" / "task4_approach_a_dataset_run.json"

# Used only by the equivalence check at the end of the notebook.
PLATFORM_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "ml" / "fare_training_dataset"
RUN_EQUIVALENCE_CHECK = os.environ.get("TASK4_SKIP_EQUIVALENCE_CHECK") != "1"

source_package = str(PROJECT_ROOT / "src")
if source_package not in sys.path:
    sys.path.insert(0, source_package)

from data_intensive_computing.spark import build_spark_session

notebook_started = time.perf_counter()
spark = build_spark_session(
    "Week4-Task4-ApproachA-Raw-Dataset",
    driver_memory="8g",
    shuffle_partitions=32,
)

stage_seconds = {}


@contextmanager
def timed(stage_name: str):
    """Record wall-clock seconds for one stage that ends with a Spark action."""
    started = time.perf_counter()
    yield
    stage_seconds[stage_name] = round(time.perf_counter() - started, 4)


print("Project root:", PROJECT_ROOT)
print("Output:", OUTPUT_PATH)
print("Spark master:", spark.sparkContext.master)
print("Equivalence check enabled:", RUN_EQUIVALENCE_CHECK)

:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a02a8f30-d023-4b44-92bf-6755ecba30d8;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

Project root: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing
Output: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/ml_raw/fare_training_dataset
Spark master: local[*]
Equivalence check enabled: True


## 1. Shared cleaning helpers

These helpers reproduce the semantics of the platform's generic validation layer:

- A field is **missing** when it is null or blank after trimming.
- A **stable hash** joins the key fields with `||`, writes nulls as `<null>`, and applies SHA-256.
- **Duplicate ranking happens over all rows, before invalid rows are removed.** A later duplicate is rejected even when the first occurrence is itself invalid. Filtering invalid rows first and deduplicating afterwards would keep that later duplicate and change the result whenever duplicates differ in a column outside the key, such as `fare_amount`.
- The platform orders duplicates by `_source_file` and `_ingested_at`. `_ingested_at` is the same `current_timestamp()` for every row in one run, so the effective order is the source file only. Ties inside one file are not ordered deterministically by the platform either.

**When can a filter move before deduplication?** Only when the filter depends exclusively on key columns. Every row in one duplicate group then receives the same filter result. This rule is used below to filter air-quality rows by state and parameter early.

In [2]:
def is_missing(column_name: str):
    return F.col(column_name).isNull() | (F.trim(F.col(column_name).cast("string")) == "")


def all_present(column_names):
    condition = F.lit(True)
    for column_name in column_names:
        condition = condition & ~is_missing(column_name)
    return condition


def stable_hash(column_names):
    return F.sha2(
        F.concat_ws(
            "||",
            *[F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in column_names],
        ),
        256,
    )


def keep_valid_first_occurrence(
    df: DataFrame,
    key_columns: list[str],
    required_columns: list[str],
    rule_conditions: list,
) -> DataFrame:
    """Keep rows that pass every rule and are the first occurrence of their key."""
    duplicate_window = Window.partitionBy(*key_columns).orderBy(F.col("_source_file"))
    ranked = df.withColumn("_duplicate_rank", F.row_number().over(duplicate_window))

    rejected = F.lit(False)
    for column_name in required_columns:
        rejected = rejected | F.coalesce(is_missing(column_name), F.lit(False))
    for condition in rule_conditions:
        # A null rule result does not reject the row, matching F.when(condition, ...).
        rejected = rejected | F.coalesce(condition, F.lit(False))
    rejected = rejected | (all_present(key_columns) & (F.col("_duplicate_rank") > 1))

    return ranked.filter(~rejected).drop("_duplicate_rank")

## 2. Taxi trips: standardize, identify, validate, deduplicate

Only the columns needed by validation, the `trip_id` key, and the Task 1 dataset are read. Dropping unused columns cannot change which rows survive.

Platform rules reproduced:

- Required: `trip_id`, `pickup_datetime`, `dropoff_datetime`, `pickup_location_id`, `dropoff_location_id`
- Rejected: drop-off before pickup; negative distance, passenger count, fare, or total amount; NaN distance or total amount
- Primary key: `trip_id`, the SHA-256 hash of seven trip fields

In [3]:
TAXI_KEY_COLUMNS = [
    "vendor_id",
    "pickup_datetime",
    "dropoff_datetime",
    "pickup_location_id",
    "dropoff_location_id",
    "trip_distance",
    "total_amount",
]

raw_taxi = (
    spark.read.parquet(str(TAXI_SOURCE))
    .withColumn("_source_file", F.input_file_name())
)

taxi = (
    raw_taxi.select(
        F.col("VendorID").cast(LongType()).alias("vendor_id"),
        F.col("tpep_pickup_datetime").cast(TimestampType()).alias("pickup_datetime"),
        F.col("tpep_dropoff_datetime").cast(TimestampType()).alias("dropoff_datetime"),
        F.col("passenger_count").cast(IntegerType()).alias("passenger_count"),
        F.col("trip_distance").cast(DoubleType()).alias("trip_distance"),
        F.col("RatecodeID").cast(LongType()).alias("rate_code_id"),
        F.col("store_and_fwd_flag").alias("store_and_fwd_flag"),
        F.col("PULocationID").cast(IntegerType()).alias("pickup_location_id"),
        F.col("DOLocationID").cast(IntegerType()).alias("dropoff_location_id"),
        F.col("fare_amount").cast(DoubleType()).alias("fare_amount"),
        F.col("total_amount").cast(DoubleType()).alias("total_amount"),
        "_source_file",
    )
    .withColumn(
        "trip_id",
        F.when(all_present(TAXI_KEY_COLUMNS), stable_hash(TAXI_KEY_COLUMNS)),
    )
    .withColumn("pickup_hour", F.date_trunc("hour", "pickup_datetime"))
    .withColumn("pickup_year", F.year("pickup_datetime").cast(IntegerType()))
)

clean_taxi = keep_valid_first_occurrence(
    taxi,
    key_columns=["trip_id"],
    required_columns=[
        "trip_id",
        "pickup_datetime",
        "dropoff_datetime",
        "pickup_location_id",
        "dropoff_location_id",
    ],
    rule_conditions=[
        F.col("dropoff_datetime") < F.col("pickup_datetime"),
        F.col("trip_distance") < 0,
        F.col("passenger_count") < 0,
        F.col("fare_amount") < 0,
        F.col("total_amount") < 0,
        F.isnan("trip_distance"),
        F.isnan("total_amount"),
    ],
).drop("_source_file")

with timed("taxi_clean"):
    raw_taxi_rows = raw_taxi.count()
    clean_taxi = clean_taxi.persist(StorageLevel.MEMORY_AND_DISK)
    clean_taxi_rows = clean_taxi.count()

print(f"Raw taxi rows:   {raw_taxi_rows:,}")
print(f"Clean taxi rows: {clean_taxi_rows:,}")
print(f"Rejected rows:   {raw_taxi_rows - clean_taxi_rows:,}")

26/10/03 15:33:19 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/raw/yellow_tripdata_2024-*.parquet.
java.io.FileNotFoundException: File /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/raw/yellow_tripdata_2024-*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSourc

Raw taxi rows:   9,554,778
Clean taxi rows: 9,417,689
Rejected rows:   137,089


## 3. Context tables: zones, weather, and air quality

The three context tables are small after preparation, so each is cached and materialized in its own timed stage. This separates their preparation cost from the main trip join.

**Zones** — required `location_id`, `borough`, `zone`; reject `location_id <= 0`; key `location_id`.

**Weather** — the hourly timestamp is built from `year`, `month`, `day`, and `hour`. Rules reject humidity outside 0–100, negative precipitation, wind direction outside 0–360, negative wind speed, and pressure outside 850–1100 hPa. Key: `observation_timestamp`.

**Air quality** — the raw CSV covers PM2.5 monitors across the whole United States. The platform validates every row, then keeps New York State (`36`) PM2.5 (`88101`) for integration. `state_code` and `parameter_code` are part of the air-quality key, so this filter can safely move before deduplication. The CSV must still be scanned completely.

In [4]:
raw_zones = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(str(ZONE_SOURCE))
    .withColumn("_source_file", F.input_file_name())
)
zones = keep_valid_first_occurrence(
    raw_zones.select(
        F.col("LocationID").cast(IntegerType()).alias("location_id"),
        F.trim("Borough").alias("borough"),
        F.trim("Zone").alias("zone"),
        "_source_file",
    ),
    key_columns=["location_id"],
    required_columns=["location_id", "borough", "zone"],
    rule_conditions=[F.col("location_id") <= 0],
).drop("_source_file")

with timed("zones_prepare"):
    zones = zones.cache()
    zone_rows = zones.count()

print(f"Zones: {zone_rows:,}")

Zones: 265


In [5]:
raw_weather = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(str(WEATHER_SOURCE))
    .withColumn("_source_file", F.input_file_name())
)
weather = (
    raw_weather.select(
        F.col("year").cast(IntegerType()).alias("year"),
        F.col("month").cast(IntegerType()).alias("month"),
        F.col("day").cast(IntegerType()).alias("day"),
        F.col("hour").cast(IntegerType()).alias("hour"),
        F.col("temp").cast(DoubleType()).alias("temp"),
        F.col("rhum").cast(IntegerType()).alias("rhum"),
        F.col("prcp").cast(DoubleType()).alias("prcp"),
        F.col("snwd").cast(DoubleType()).alias("snwd"),
        F.col("wdir").cast(IntegerType()).alias("wdir"),
        F.col("wspd").cast(DoubleType()).alias("wspd"),
        F.col("pres").cast(DoubleType()).alias("pres"),
        F.col("coco").cast(IntegerType()).alias("coco"),
        "_source_file",
    )
    .withColumn(
        "observation_timestamp",
        F.make_timestamp("year", "month", "day", "hour", F.lit(0), F.lit(0)).cast(TimestampType()),
    )
)
weather = keep_valid_first_occurrence(
    weather,
    key_columns=["observation_timestamp"],
    required_columns=["observation_timestamp"],
    rule_conditions=[
        (F.col("rhum") < 0) | (F.col("rhum") > 100),
        F.col("prcp") < 0,
        (F.col("wdir") < 0) | (F.col("wdir") > 360),
        F.col("wspd") < 0,
        (F.col("pres") < 850) | (F.col("pres") > 1100),
    ],
).select(
    F.col("observation_timestamp").alias("weather_observation_timestamp"),
    F.col("temp").alias("weather_temperature_c"),
    F.col("rhum").alias("weather_relative_humidity_pct"),
    F.col("prcp").alias("weather_precipitation_mm"),
    F.col("snwd").alias("weather_snow_depth_mm"),
    F.col("wspd").alias("weather_wind_speed_kmh"),
    F.col("pres").alias("weather_pressure_hpa"),
    F.col("coco").alias("weather_condition_code"),
)

with timed("weather_prepare"):
    weather = weather.cache()
    weather_rows = weather.count()

print(f"Weather hours: {weather_rows:,}")

Weather hours: 8,783


In [6]:
AIR_QUALITY_KEY_COLUMNS = [
    "state_code",
    "county_code",
    "site_num",
    "parameter_code",
    "poc",
    "observation_timestamp",
    "method_code",
]

raw_air_quality = (
    spark.read.option("header", "true").option("inferSchema", "false")
    .csv(str(AIR_QUALITY_SOURCE))
    .withColumn("_source_file", F.input_file_name())
)

air_quality = (
    raw_air_quality
    # Safe early filter: both columns belong to the deduplication key.
    .filter((F.col("State Code") == "36") & (F.col("Parameter Code") == "88101"))
    .select(
        F.col("State Code").alias("state_code"),
        F.col("County Code").alias("county_code"),
        F.col("Site Num").alias("site_num"),
        F.col("Parameter Code").alias("parameter_code"),
        F.col("POC").alias("poc"),
        F.col("Method Code").alias("method_code"),
        F.col("Latitude").cast(DoubleType()).alias("latitude"),
        F.col("Longitude").cast(DoubleType()).alias("longitude"),
        F.col("Sample Measurement").cast(DoubleType()).alias("sample_measurement"),
        F.to_timestamp(
            F.concat_ws(" ", "Date Local", "Time Local"), "yyyy-MM-dd HH:mm"
        ).alias("observation_timestamp"),
        F.to_timestamp(
            F.concat_ws(" ", "Date GMT", "Time GMT"), "yyyy-MM-dd HH:mm"
        ).alias("observation_timestamp_utc"),
        "_source_file",
    )
    .withColumn(
        "station_id",
        F.concat_ws("-", "state_code", "county_code", "site_num", "parameter_code", "poc"),
    )
    .withColumn(
        "observation_id",
        F.when(all_present(AIR_QUALITY_KEY_COLUMNS), stable_hash(AIR_QUALITY_KEY_COLUMNS)),
    )
)

air_quality = keep_valid_first_occurrence(
    air_quality,
    key_columns=["observation_id"],
    required_columns=[
        "observation_id",
        "station_id",
        "parameter_code",
        "observation_timestamp",
        "observation_timestamp_utc",
        "sample_measurement",
    ],
    rule_conditions=[
        (F.col("latitude") < -90) | (F.col("latitude") > 90),
        (F.col("longitude") < -180) | (F.col("longitude") > 180),
        F.col("sample_measurement") < 0,
        F.isnan("sample_measurement"),
    ],
)

# New York State county codes for the five NYC boroughs.
nyc_counties = spark.createDataFrame(
    [
        ("005", "Bronx"),
        ("047", "Brooklyn"),
        ("061", "Manhattan"),
        ("081", "Queens"),
        ("085", "Staten Island"),
    ],
    ["county_code", "air_quality_borough"],
)

air_quality_hourly = (
    air_quality
    .join(F.broadcast(nyc_counties), "county_code", "inner")
    .groupBy("air_quality_borough", "observation_timestamp")
    .agg(F.avg("sample_measurement").alias("air_quality_pm25_mean"))
    .withColumnRenamed("observation_timestamp", "air_quality_observation_timestamp")
)

with timed("air_quality_prepare"):
    air_quality_hourly = air_quality_hourly.cache()
    air_quality_hour_rows = air_quality_hourly.count()

print(f"Borough-hour PM2.5 rows: {air_quality_hour_rows:,}")

Borough-hour PM2.5 rows: 26,348


## 4. Integrate trips with context

All joins are left joins, so context never removes a trip, which matches the platform. The Task 1 modeling window is applied before the joins. This is safe: the window depends only on trip columns, deduplication has already happened, and left joins do not change the set of trips.

In [7]:
TRAIN_START = "2024-01-01"
VALIDATION_START = "2024-03-01"
TEST_START = "2024-03-16"
TEST_END = "2024-04-01"

modeling_trips = (
    clean_taxi
    .filter(F.col("pickup_year") == 2024)
    .filter(
        (F.col("pickup_datetime") >= F.to_timestamp(F.lit(TRAIN_START)))
        & (F.col("pickup_datetime") < F.to_timestamp(F.lit(TEST_END)))
    )
)

pickup_zones = F.broadcast(
    zones.select(
        F.col("location_id").alias("pickup_zone_location_id"),
        F.col("zone").alias("pickup_zone"),
        F.col("borough").alias("pickup_borough"),
    )
)
dropoff_zones = F.broadcast(
    zones.select(
        F.col("location_id").alias("dropoff_zone_location_id"),
        F.col("zone").alias("dropoff_zone"),
        F.col("borough").alias("dropoff_borough"),
    )
)

integrated = (
    modeling_trips
    .join(pickup_zones, F.col("pickup_location_id") == F.col("pickup_zone_location_id"), "left")
    .drop("pickup_zone_location_id")
    .join(dropoff_zones, F.col("dropoff_location_id") == F.col("dropoff_zone_location_id"), "left")
    .drop("dropoff_zone_location_id")
    .join(
        F.broadcast(weather),
        F.col("pickup_hour") == F.col("weather_observation_timestamp"),
        "left",
    )
    .join(
        F.broadcast(air_quality_hourly),
        (F.col("pickup_borough") == F.col("air_quality_borough"))
        & (F.col("pickup_hour") == F.col("air_quality_observation_timestamp")),
        "left",
    )
)

## 5. Task 1 logic: population, label, features, and split

This section repeats the Week 4 Task 1 rules exactly: borough eligibility is computed from Train only, the label is the non-negative base fare, and the splits are chronological.

In [8]:
MIN_TRAIN_TRIP_COUNT = 10_000
NYC_BOROUGH_CANDIDATES = ["Bronx", "Brooklyn", "Manhattan", "Queens", "Staten Island"]
EXPECTED_SELECTED_BOROUGHS = {"Bronx", "Brooklyn", "Manhattan", "Queens"}

with timed("borough_selection"):
    train_borough_counts = (
        integrated
        .filter(F.col("pickup_datetime") < F.to_timestamp(F.lit(VALIDATION_START)))
        .filter(F.col("pickup_borough").isin(NYC_BOROUGH_CANDIDATES))
        .groupBy("pickup_borough")
        .agg(F.count("*").alias("train_trip_count"))
        .collect()
    )

selected_boroughs = sorted(
    row["pickup_borough"]
    for row in train_borough_counts
    if row["train_trip_count"] >= MIN_TRAIN_TRIP_COUNT
)
assert set(selected_boroughs) == EXPECTED_SELECTED_BOROUGHS, selected_boroughs
print("Train borough counts:", {r["pickup_borough"]: r["train_trip_count"] for r in train_borough_counts})
print("Selected boroughs:", selected_boroughs)

Train borough counts: {'Staten Island': 122, 'Bronx': 13757, 'Brooklyn': 52551, 'Manhattan': 5293115, 'Queens': 510939}
Selected boroughs: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens']


In [9]:
training_dataset = (
    integrated
    .filter(F.col("pickup_borough").isin(selected_boroughs))
    .filter(F.col("fare_amount").isNotNull() & (F.col("fare_amount") >= 0))
    .filter(F.col("trip_distance").isNotNull() & (F.col("trip_distance") >= 0))
    .withColumn("pickup_hour_of_day", F.hour("pickup_datetime"))
    .withColumn("pickup_day_of_week", F.dayofweek("pickup_datetime"))
    .withColumn("pickup_month_of_year", F.month("pickup_datetime"))
    .withColumn(
        "is_weekend",
        F.when(F.dayofweek("pickup_datetime").isin(1, 7), F.lit(1.0)).otherwise(F.lit(0.0)),
    )
    .withColumn("pickup_hour_sin", F.sin(2 * F.lit(math.pi) * F.col("pickup_hour_of_day") / F.lit(24.0)))
    .withColumn("pickup_hour_cos", F.cos(2 * F.lit(math.pi) * F.col("pickup_hour_of_day") / F.lit(24.0)))
    .withColumn("pickup_dow_sin", F.sin(2 * F.lit(math.pi) * (F.col("pickup_day_of_week") - 1) / F.lit(7.0)))
    .withColumn("pickup_dow_cos", F.cos(2 * F.lit(math.pi) * (F.col("pickup_day_of_week") - 1) / F.lit(7.0)))
    .withColumn("air_quality_missing", F.col("air_quality_pm25_mean").isNull().cast("double"))
    .withColumn(
        "split",
        F.when(F.col("pickup_datetime") < F.to_timestamp(F.lit(VALIDATION_START)), F.lit("train"))
        .when(F.col("pickup_datetime") < F.to_timestamp(F.lit(TEST_START)), F.lit("validation"))
        .otherwise(F.lit("test")),
    )
    .select(
        "trip_id",
        "pickup_datetime",
        "split",
        F.col("fare_amount").alias("label"),
        "pickup_zone",
        "dropoff_zone",
        "pickup_borough",
        "dropoff_borough",
        "trip_distance",
        "passenger_count",
        "vendor_id",
        "rate_code_id",
        "store_and_fwd_flag",
        "pickup_hour_of_day",
        "pickup_day_of_week",
        "pickup_month_of_year",
        "is_weekend",
        "pickup_hour_sin",
        "pickup_hour_cos",
        "pickup_dow_sin",
        "pickup_dow_cos",
        "weather_temperature_c",
        "weather_relative_humidity_pct",
        "weather_precipitation_mm",
        "weather_snow_depth_mm",
        "weather_wind_speed_kmh",
        "weather_pressure_hpa",
        "weather_condition_code",
        "air_quality_pm25_mean",
        "air_quality_missing",
    )
)

training_dataset.printSchema()

root
 |-- trip_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- split: string (nullable = false)
 |-- label: double (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- dropoff_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- dropoff_borough: string (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- vendor_id: long (nullable = true)
 |-- rate_code_id: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pickup_hour_of_day: integer (nullable = true)
 |-- pickup_day_of_week: integer (nullable = true)
 |-- pickup_month_of_year: integer (nullable = true)
 |-- is_weekend: double (nullable = false)
 |-- pickup_hour_sin: double (nullable = true)
 |-- pickup_hour_cos: double (nullable = true)
 |-- pickup_dow_sin: double (nullable = true)
 |-- pickup_dow_cos: double (nullable = true)
 |-- weather_temperature_c: double (nullabl

## 6. Write the training Delta table and the run manifest

In [10]:
with timed("write_dataset"):
    (
        training_dataset.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("split")
        .save(str(OUTPUT_PATH))
    )

clean_taxi.unpersist()
dataset_seconds = round(time.perf_counter() - notebook_started, 4)

split_counts = {
    row["split"]: row["count"]
    for row in spark.read.format("delta").load(str(OUTPUT_PATH)).groupBy("split").count().collect()
}

spark_conf = spark.sparkContext.getConf()
run_manifest = {
    "run_id": str(uuid.uuid4()),
    "recorded_at": datetime.now(timezone.utc).isoformat(),
    "approach": "raw",
    "output_path": str(OUTPUT_PATH),
    "row_counts": {
        "raw_taxi": raw_taxi_rows,
        "clean_taxi": clean_taxi_rows,
        "zones": zone_rows,
        "weather_hours": weather_rows,
        "air_quality_borough_hours": air_quality_hour_rows,
        "splits": split_counts,
    },
    "selected_boroughs": selected_boroughs,
    "timings_seconds": {
        **stage_seconds,
        "session_start_to_dataset_written": dataset_seconds,
    },
    "spark": {
        "version": spark.version,
        "master": spark_conf.get("spark.master", ""),
        "driver_memory": spark_conf.get("spark.driver.memory", ""),
        "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
        "session_time_zone": spark.conf.get("spark.sql.session.timeZone"),
    },
}

RUN_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
RUN_MANIFEST_PATH.write_text(json.dumps(run_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(run_manifest["timings_seconds"], indent=2))
print("Split counts:", split_counts)

26/10/03 15:33:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


{
  "taxi_clean": 12.0995,
  "zones_prepare": 0.1327,
  "weather_prepare": 0.2208,
  "air_quality_prepare": 3.2088,
  "borough_selection": 0.8153,
  "write_dataset": 6.7639,
  "session_start_to_dataset_written": 28.2801
}
Split counts: {'train': 5870362, 'test': 1783894, 'validation': 1726201}


## 7. Equivalence check against Approach B

This check is the success criterion for a faithful reproduction. It compares:

1. the schema, including column order;
2. the row count of every split;
3. the set of `trip_id` values in both directions; and
4. every column for matching `trip_id` values, both exactly and with a `1e-9` tolerance for doubles. Floating-point averages can differ in the last bits when Spark sums values in a different order.

Set the environment variable `TASK4_SKIP_EQUIVALENCE_CHECK=1` to skip this section during timing runs.

In [11]:
if RUN_EQUIVALENCE_CHECK:
    approach_a = spark.read.format("delta").load(str(OUTPUT_PATH))
    approach_b = spark.read.format("delta").load(str(PLATFORM_DATASET_PATH))

    a_schema = [(f.name, f.dataType.simpleString()) for f in approach_a.schema.fields]
    b_schema = [(f.name, f.dataType.simpleString()) for f in approach_b.schema.fields]
    print("Schema identical:", a_schema == b_schema)
    if a_schema != b_schema:
        print("Only in A:", sorted(set(a_schema) - set(b_schema)))
        print("Only in B:", sorted(set(b_schema) - set(a_schema)))

    b_split_counts = {r["split"]: r["count"] for r in approach_b.groupBy("split").count().collect()}
    print("Split counts A:", split_counts)
    print("Split counts B:", b_split_counts)

    a_ids = approach_a.select("trip_id")
    b_ids = approach_b.select("trip_id")
    only_in_a = a_ids.exceptAll(b_ids).count()
    only_in_b = b_ids.exceptAll(a_ids).count()
    print(f"trip_id only in A: {only_in_a:,}")
    print(f"trip_id only in B: {only_in_b:,}")

Schema identical: True
Split counts A: {'train': 5870362, 'test': 1783894, 'validation': 1726201}
Split counts B: {'train': 5870362, 'validation': 1726201, 'test': 1783894}


trip_id only in A: 0
trip_id only in B: 0


In [12]:
if RUN_EQUIVALENCE_CHECK:
    TOLERANCE = 1e-9
    compared_columns = [c for c in approach_b.columns if c != "trip_id"]
    double_columns = {f.name for f in approach_b.schema.fields if f.dataType.simpleString() == "double"}

    joined = approach_a.alias("a").join(approach_b.alias("b"), "trip_id", "inner")

    aggregations = []
    for column_name in compared_columns:
        a_col, b_col = F.col(f"a.{column_name}"), F.col(f"b.{column_name}")
        exact_mismatch = ~a_col.eqNullSafe(b_col)
        aggregations.append(F.sum(exact_mismatch.cast("long")).alias(f"{column_name}__exact"))
        if column_name in double_columns:
            tolerant_mismatch = (
                F.when(a_col.isNull() & b_col.isNull(), F.lit(False))
                .when(a_col.isNull() | b_col.isNull(), F.lit(True))
                .otherwise(F.abs(a_col - b_col) > TOLERANCE)
            )
            aggregations.append(F.sum(tolerant_mismatch.cast("long")).alias(f"{column_name}__tolerant"))

    mismatch_counts = joined.agg(*aggregations).first().asDict()
    nonzero = {k: v for k, v in mismatch_counts.items() if v}
    print("Matched rows compared:", f"{joined.count():,}")
    print("Columns with mismatches:", nonzero if nonzero else "none")

    exact_equal = (
        a_schema == b_schema
        and split_counts == b_split_counts
        and only_in_a == 0
        and only_in_b == 0
        and not nonzero
    )
    tolerant_equal = (
        a_schema == b_schema
        and split_counts == b_split_counts
        and only_in_a == 0
        and only_in_b == 0
        and not any(v for k, v in mismatch_counts.items() if k.endswith("__tolerant") or (k.endswith("__exact") and k[:-7] not in double_columns))
    )
    print("Exactly identical to Approach B:", exact_equal)
    print("Identical within floating-point tolerance:", tolerant_equal)

Matched rows compared: 9,380,457
Columns with mismatches: none
Exactly identical to Approach B: True
Identical within floating-point tolerance: True
